# Notebook 5: Loan Default Prediction (Complex - Full ML Pipeline)
End-to-end ML workflow: SQL feature engineering, Python EDA, model training, evaluation, and results written back to Snowflake.

### SQL Feature Engineering - Build ML Dataset
This query builds the full feature set for the machine learning model. A CTE first aggregates payment behaviour per loan (on-time, missed, partial counts, days late, and payment ratio). These payment features are then joined with loan, customer, and application data to produce a single flat dataset. Key engineered features include loan-to-income ratio, borrower age derived from date of birth, and a binary IS_DEFAULT target label for model training.

In [ ]:
%%sql -r ml_features
WITH payment_features AS (
    SELECT p.LOAN_ID,
           COUNT(*) AS PAYMENT_COUNT,
           SUM(CASE WHEN PAYMENT_STATUS = 'ON_TIME' THEN 1 ELSE 0 END) AS ON_TIME_PAYMENTS,
           SUM(CASE WHEN PAYMENT_STATUS = 'MISSED' THEN 1 ELSE 0 END) AS MISSED_PAYMENTS,
           SUM(CASE WHEN PAYMENT_STATUS = 'PARTIAL' THEN 1 ELSE 0 END) AS PARTIAL_PAYMENTS,
           MAX(DAYS_LATE) AS MAX_DAYS_LATE,
           AVG(DAYS_LATE) AS AVG_DAYS_LATE,
           ROUND(SUM(AMOUNT_PAID) / NULLIF(SUM(AMOUNT_DUE), 0), 3) AS PAYMENT_RATIO
    FROM LOAN_DB.LENDING.PAYMENTS p
    GROUP BY p.LOAN_ID
)
SELECT l.LOAN_ID,
       l.LOAN_TYPE, l.LOAN_AMOUNT, l.INTEREST_RATE, l.TERM_MONTHS,
       l.MONTHLY_PAYMENT,
       CASE WHEN l.LOAN_STATUS IN ('DEFAULT', 'DELINQUENT') THEN 1 ELSE 0 END AS IS_DEFAULT,
       c.CREDIT_SCORE, c.ANNUAL_INCOME, c.EMPLOYMENT_STATUS,
       DATEDIFF('year', c.DATE_OF_BIRTH, CURRENT_DATE()) AS AGE,
       a.DEBT_TO_INCOME_RATIO,
       ROUND(l.LOAN_AMOUNT / NULLIF(c.ANNUAL_INCOME, 0), 3) AS LOAN_TO_INCOME_RATIO,
       COALESCE(pf.PAYMENT_COUNT, 0) AS PAYMENT_COUNT,
       COALESCE(pf.ON_TIME_PAYMENTS, 0) AS ON_TIME_PAYMENTS,
       COALESCE(pf.MISSED_PAYMENTS, 0) AS MISSED_PAYMENTS,
       COALESCE(pf.PARTIAL_PAYMENTS, 0) AS PARTIAL_PAYMENTS,
       COALESCE(pf.MAX_DAYS_LATE, 0) AS MAX_DAYS_LATE,
       COALESCE(pf.AVG_DAYS_LATE, 0) AS AVG_DAYS_LATE,
       COALESCE(pf.PAYMENT_RATIO, 1.0) AS PAYMENT_RATIO
FROM LOAN_DB.LENDING.LOANS l
JOIN LOAN_DB.LENDING.CUSTOMERS c ON l.CUSTOMER_ID = c.CUSTOMER_ID
JOIN LOAN_DB.LENDING.LOAN_APPLICATIONS a ON l.CUSTOMER_ID = a.CUSTOMER_ID
LEFT JOIN payment_features pf ON l.LOAN_ID = pf.LOAN_ID

### Explore ML Dataset
This cell performs a quick sanity check on the dataset before modelling begins. It prints the total number of rows and columns, shows how many loans are labelled as defaulted vs good, and displays the data types of each feature. This helps confirm the dataset loaded correctly and gives an early view of any class imbalance in the target variable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = ml_features.copy()
print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['IS_DEFAULT'].value_counts())
print(f"\nDefault rate: {df['IS_DEFAULT'].mean()*100:.1f}%")
print(f"\nFeature types:")
print(df.dtypes)

### Feature Distribution EDA
This cell plots overlapping histograms for nine key numeric features, split by whether the loan defaulted or not. Comparing the green (good) and red (default) distributions side by side reveals which features have the clearest separation between the two classes — an important step in understanding which variables are likely to be most predictive in the model.

In [ ]:
numeric_cols = ['LOAN_AMOUNT', 'INTEREST_RATE', 'TERM_MONTHS', 'CREDIT_SCORE',
                'ANNUAL_INCOME', 'DEBT_TO_INCOME_RATIO', 'LOAN_TO_INCOME_RATIO',
                'PAYMENT_RATIO', 'MAX_DAYS_LATE']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i, col in enumerate(numeric_cols):
    ax = axes[i//3, i%3]
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = df[df['IS_DEFAULT'] == label][col]
        ax.hist(subset, bins=15, alpha=0.6, color=color,
                label='Good' if label == 0 else 'Default')
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Default Status', fontsize=14)
plt.tight_layout()
plt.show()

### Correlation Heatmap
This cell computes and visualises the pairwise correlation matrix for the most important model features, including the IS_DEFAULT target. Strong positive or negative correlations with IS_DEFAULT highlight the most predictive features, while correlations between features can reveal multicollinearity that may affect certain model types.

In [ ]:
corr_cols = ['IS_DEFAULT', 'CREDIT_SCORE', 'INTEREST_RATE', 'DEBT_TO_INCOME_RATIO',
             'LOAN_TO_INCOME_RATIO', 'PAYMENT_RATIO', 'MAX_DAYS_LATE', 'MISSED_PAYMENTS']

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

### Prepare Features & Train/Test Split
This cell prepares the data for model training. Categorical columns (loan type and employment status) are label-encoded into numeric form. All features are then standardised using StandardScaler to ensure no single feature dominates due to its scale. The dataset is split 70/30 into training and test sets using stratified sampling to preserve the default rate in both splits.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

model_df = df.copy()
le = LabelEncoder()
model_df['LOAN_TYPE_ENC'] = le.fit_transform(model_df['LOAN_TYPE'])
model_df['EMPLOYMENT_ENC'] = le.fit_transform(model_df['EMPLOYMENT_STATUS'])

feature_cols = ['LOAN_AMOUNT', 'INTEREST_RATE', 'TERM_MONTHS', 'MONTHLY_PAYMENT',
                'CREDIT_SCORE', 'ANNUAL_INCOME', 'DEBT_TO_INCOME_RATIO',
                'LOAN_TO_INCOME_RATIO', 'AGE', 'PAYMENT_COUNT', 'ON_TIME_PAYMENTS',
                'MISSED_PAYMENTS', 'PARTIAL_PAYMENTS', 'MAX_DAYS_LATE',
                'AVG_DAYS_LATE', 'PAYMENT_RATIO', 'LOAN_TYPE_ENC', 'EMPLOYMENT_ENC']

X = model_df[feature_cols].fillna(0)
y = model_df['IS_DEFAULT']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train default rate: {y_train.mean()*100:.1f}%")
print(f"Test default rate: {y_test.mean()*100:.1f}%")

### Train & Compare 3 Models
This cell trains three different classification models — Logistic Regression, Random Forest, and Gradient Boosting — on the training set and evaluates each against the test set. For every model it prints a full classification report (precision, recall, F1) and the AUC score, making it easy to compare their performance and select the best one for scoring.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba) if len(y_test.unique()) > 1 else 0.0
    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_proba': y_proba, 'auc': auc
    }
    print(f"\n{'='*50}")
    print(f"{name} (AUC: {auc:.3f})")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, zero_division=0))

### Model Evaluation Plots
This cell produces three evaluation charts side by side. The ROC curve compares all three models visually, with AUC scores in the legend. The feature importance bar chart shows the top 10 most influential features from the best-performing model. The confusion matrix shows how many loans the best model correctly and incorrectly classified as good or default.

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})")
axes[0].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[0].set_title('ROC Curves')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

best_name = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]['model']
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
    importances.tail(10).plot(kind='barh', ax=axes[1], color='#3498db')
    axes[1].set_title(f'Top 10 Features ({best_name})')

cm = confusion_matrix(y_test, results[best_name]['y_pred'])
im = axes[2].imshow(cm, cmap='Blues')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[2].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)
axes[2].set_title(f'Confusion Matrix ({best_name})')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_xticks([0,1])
axes[2].set_yticks([0,1])
axes[2].set_xticklabels(['Good', 'Default'])
axes[2].set_yticklabels(['Good', 'Default'])

plt.tight_layout()
plt.show()
print(f"\nBest model: {best_name} with AUC = {results[best_name]['auc']:.3f}")

### Score All Loans with Best Model
This cell applies the best-performing model to the entire dataset to generate a default probability score for every loan. Loans with a probability of 0.5 or above are flagged as predicted defaults. The output is sorted from highest to lowest risk, giving a ranked list of loans that can be used for prioritisation and collections strategy.

In [ ]:
model_df['DEFAULT_PROBABILITY'] = best_model.predict_proba(
    scaler.transform(model_df[feature_cols].fillna(0))
)[:, 1]
model_df['PREDICTED_DEFAULT'] = (model_df['DEFAULT_PROBABILITY'] >= 0.5).astype(int)

scoring_output = model_df[['LOAN_ID', 'LOAN_TYPE', 'LOAN_AMOUNT', 'CREDIT_SCORE',
                            'IS_DEFAULT', 'DEFAULT_PROBABILITY', 'PREDICTED_DEFAULT']]
scoring_output = scoring_output.sort_values('DEFAULT_PROBABILITY', ascending=False)
scoring_output

### Create Risk Scores Table
This SQL statement creates a dedicated table in Snowflake to store the model's output scores. It is defined with IF NOT EXISTS so it is safe to run multiple times without error. The table schema includes the loan details, the actual default label, the model's predicted probability, and a timestamp recording when the scores were generated.

In [ ]:
%%sql -r create_result
CREATE TABLE IF NOT EXISTS LOAN_DB.LENDING.LOAN_RISK_SCORES (
    LOAN_ID INT,
    LOAN_TYPE VARCHAR(30),
    LOAN_AMOUNT NUMBER(12,2),
    CREDIT_SCORE INT,
    ACTUAL_DEFAULT INT,
    DEFAULT_PROBABILITY FLOAT,
    PREDICTED_DEFAULT INT,
    SCORED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)

### Write Risk Scores Back to Snowflake
This cell uses the active Snowpark session to write the scored loan dataset back into the Snowflake table created above. The data is written in overwrite mode so each run produces a fresh, up-to-date set of scores. A confirmation message prints the number of rows successfully written.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

output_df = scoring_output.rename(columns={'IS_DEFAULT': 'ACTUAL_DEFAULT'})
snow_df = session.create_dataframe(output_df)
snow_df.write.mode('overwrite').save_as_table('LOAN_DB.LENDING.LOAN_RISK_SCORES')

print(f"Successfully wrote {len(output_df)} risk scores to LOAN_DB.LENDING.LOAN_RISK_SCORES")

### Verify: Top 10 Riskiest Loans
This final query reads directly from the LOAN_RISK_SCORES table in Snowflake to confirm the data was written successfully. It returns the 10 loans with the highest predicted default probability, providing a quick end-to-end verification that the full ML pipeline — from feature engineering through to Snowflake write-back — has completed correctly.

In [ ]:
%%sql -r top_risky_loans
SELECT LOAN_ID, LOAN_TYPE, LOAN_AMOUNT, CREDIT_SCORE,
       ACTUAL_DEFAULT, ROUND(DEFAULT_PROBABILITY, 3) AS DEFAULT_PROB, PREDICTED_DEFAULT
FROM LOAN_DB.LENDING.LOAN_RISK_SCORES
ORDER BY DEFAULT_PROBABILITY DESC
LIMIT 10